In [1]:
!pip install mamba-ssm[causal-conv1d]
!pip install triton

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.5.0.post8-cp310-cp310-linux_x86_64.whl size=103960374 sha256=ec956a2d0fa48bd16a8dd5be9ad1c03db88565ea6ee2679e362940fef659284b
  Stored in directory: /root/.cache/pip/wheels/75/ef/0a/d9abf869acdd5fc07f403f4d8dd9db650cd66e81528a907941
  Created wheel for mamba-ssm: filename=mamba_ssm-2.2.4-cp310-cp310-linux_x86_64.whl size=323655716 sha256=0f4b4e95ce6271534b916f819bd33c1283d96208fbaac3f62a3c9777a3501187
  Stored in directory: /root/.cache/pip/wheels/aa/af/c7/fb77bfcd94bd3e052545033449d8c47dc97222d79c39c5bc67
Successfully built causal-conv1d mamba-ssm
  Using cached triton-3.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
Using cached triton-3.2.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm.modules.mamba_simple import Mamba

class Add_Norm(nn.Module):
    def __init__(self, d_model, dropout, residual, drop_flag=1):
        super(Add_Norm, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        self.residual = residual
        self.drop_flag = drop_flag
    
    def forward(self, new, old):
        new = self.dropout(new) if self.drop_flag else new
        return self.norm(old + new) if self.residual else self.norm(new)

class BimambaEncoderLayer(nn.Module):
    def __init__(self, 
                 d_model,
                 d_conv,
                 d_state,
                 expand,
                 dropout=0.2,
                 d_ff=256, 
                 activation="relu", 
                 residual=1):
        super(BimambaEncoderLayer, self).__init__()
        self.d_model=d_model
        self.d_ff=d_ff
        self.d_conv=d_conv
        self.d_state=d_state
        self.expand=expand

        self.mamba_forward=Mamba(
            d_model=self.d_model,
            d_state=self.d_state,
            d_conv=self.d_conv,
            expand=self.expand,
        )
        self.addnorm_for=Add_Norm(d_model,dropout,residual=0,drop_flag=0)
        self.mamba_backward=Mamba(
            d_model=self.d_model,
            d_state=self.d_state,
            d_conv=self.d_conv,
            expand=self.expand,
        ) 
        self.addnorm_back=Add_Norm(d_model,dropout,residual=0,drop_flag=0)
        self.addnorm_output=Add_Norm(d_model,dropout,residual=1,drop_flag=0)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.addnorm_ffn = Add_Norm(d_model, dropout, residual, drop_flag=1)

    def forward(self, x):
        # [B, S, D]
        output_forward = self.mamba_forward(x)
        output_forward = self.addnorm_for(output_forward, x)
        output_backward = self.mamba_backward(x.flip(dims=[1])).flip(dims=[1])
        output_backward = self.addnorm_back(output_backward, x)
        output = output_forward + output_backward
        output = self.addnorm_output(output,x)
        temp = output
        output = self.feed_forward(output)
        output = self.addnorm_ffn(output, temp)
        return output


def createLayer(
    d_model,
    d_conv,
    d_state,
    expand,
    dropout
):
    layer = BimambaEncoderLayer(
        d_model,
        d_conv,
        d_state,
        expand,
        dropout
    )
    return layer

class Encoder(nn.Module):
    def __init__(
        self, 
        d_model,
        d_conv,
        d_state,
        expand,
        dropout,
        n_layer,
        vocab_size,
    ):
        super(Encoder, self).__init__()
        self.d_model=d_model
        self.d_conv=d_conv
        self.d_state=d_state
        self.expand=expand
        self.droput=dropout
        self.n_layer=n_layer
        self.vocab_size=vocab_size
        self.embedding = nn.Embedding(vocab_size,d_model)
        self.layers = nn.ModuleList(
            [
                createLayer(
                    d_model,
                    d_conv,
                    d_state,
                    expand,
                    dropout
                )
                for _ in range(n_layer)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
    
    def forward(self,x):
        x=self.embedding(x)
        for layers in self.layers:
            x=layers(x)

        x=self.final_norm(x)

        return x
        

In [3]:
class DecoderLayer(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super(DecoderLayer,self).__init__()
        self.self_attention = nn.MultiheadAttention(d_model,n_heads,dropout=dropout)
        self.encoder_attention = nn.MultiheadAttention(d_model,n_heads,dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, target, encoder_output, trg_mask=None, src_mask=None):
        target_transpose=target.transpose(0,1)
        # Self attention
        _target, _ = self.self_attention(target_transpose,
                                         target_transpose,
                                         target_transpose,
                                         attn_mask=trg_mask)
        _target=_target.transpose(0,1)
        target = self.norm1(target + self.dropout1(_target))
        # Encoder attention (cross attention)
        target_transpose=target.transpose(0,1)
        encoder_output_transpose=encoder_output.transpose(0,1)
        target, attn_weights = self.encoder_attention(
            target_transpose,
            encoder_output_transpose,
            encoder_output_transpose,
            attn_mask=src_mask
        )
        target=target.transpose(0,1)
        target = self.norm2(target + self.dropout2(_target))
        
        # Feed forward
        _target = self.ffn(target)
        target = self.norm3(target + self.dropout3(_target))
        
        return target

class Decoder(nn.Module):
    def __init__(self,vocab_size,d_model,n_heads,n_layer,dropout):
        super(Decoder,self).__init__()
        self.d_model=d_model
        self.vocab_size=vocab_size
        self.n_heads=n_heads
        self.n_layer=n_layer
        self.dropout=dropout

        self.embedding=nn.Embedding(vocab_size,d_model)
        self.position_encodding=nn.Parameter(torch.randn(1, 1, d_model))
        self.dropout=nn.Dropout(dropout)

        self.decoder_layers=nn.ModuleList([
            DecoderLayer(
                d_model,
                n_heads,
                dropout
            )
            for _ in range(n_layer)
        ])

        self.logits=nn.Linear(d_model,vocab_size)

    def forward(self,target,encoder_output,trg_mask=None,src_mask=None):
        
        Batch,length=target.shape
        target=self.embedding(target)*torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))
        target+=self.position_encodding[:,:length,:]
        target=self.dropout(target)

        for layers in self.decoder_layers:
            target=layers(target,encoder_output,trg_mask,src_mask)

        output=self.logits(target)
        return output
        

In [4]:
d_model = 512
d_state = 64
d_conv=4
expand=2
n_heads=8
dropout=0.2

vocab_size=50000
encoder_n_layer=6
decoder_n_layer=6
model = Encoder(
    d_model=d_model,
    d_conv=d_conv,
    d_state=d_state,
    expand=expand,
    dropout=dropout,
    n_layer=encoder_n_layer,
    vocab_size=vocab_size
    
).to('cuda')

batch, length = 2, 64
x = torch.randint(0,49999,(batch, length)).to("cuda")
encoder_output=model(x)
print(encoder_output.shape)

decoder = Decoder(
    d_model=d_model,
    n_heads=n_heads,
    vocab_size=vocab_size,
    n_layer=decoder_n_layer,
    dropout=dropout
).to('cuda')
length=100
target = torch.randint(0,49999,(batch, length)).to("cuda")

decoder_output = decoder(target,encoder_output)

print(decoder_output.shape)

torch.Size([2, 64, 512])
torch.Size([2, 100, 50000])
